# CAB320 Assignment 2 - Transfer Learning
Anthony Vanderkop, Thierry Peynot, Frederic Maire (Jupyter Notebook template: 2025)


## Instructions:
The functions and classes defined in this module will be called by the marker without modification. 
You should complete the functions and classes according to their specified interfaces.

No partial marks will be awarded for functions that do not meet the specifications of the interfaces.


In [1]:
### LIBRARY IMPORTS ###
import os
import numpy as np
import keras.applications as ka
import keras


## Task 1
Implement the my_team()function 

In [2]:
def my_team():
    '''
    Return the list of the team members of this assignment submission as a list
    of triplet of the form (student_number, first_name, last_name)
    
    '''
    return [
        ('n11285915', 'Nour', 'Spear'),
        ('n11323442', 'Joshua', 'Oates'),
        ('n12116785', 'Joshua', 'George'),
    ]

In [3]:
my_team()

[('n11285915', 'Nour', 'Spear'),
 ('n11323442', 'Joshua', 'Oates'),
 ('n12116785', 'Joshua', 'George')]

## Task 2
Download the small_flower_dataset from Canvas and load the data

In [4]:
def load_data(path):
    '''
    Load in the dataset from its home path. Path should be a string of the path
    to the home directory the dataset is found in. Should return a numpy array
    with paired images and class labels.
    
    Insert a more detailed description here.

    Returns:
        X: numpy array of shape (n_samples, 224, 224, 3)
        Y: numpy array of shape (n_samples,)
    '''
    # ensures imported path is in the correct file system format
    data_root = os.fspath(path)
    nested_root = os.path.join(data_root, 'small_flower_dataset')

    # handles the extra folder level created when the zip file is unzipped
    if os.path.isdir(nested_root):
        data_root = nested_root

    # stops the function if the dataset folder cannot be found
    if not os.path.isdir(data_root):
        raise FileNotFoundError(f'Dataset folder not found: {path}')

    # loads the images from each class folder and resizes them for MobileNetV2
    image_dataset = keras.utils.image_dataset_from_directory(
        data_root,
        labels='inferred',
        label_mode='int',
        color_mode='rgb',
        batch_size=None,
        image_size=(224, 224),
        shuffle=False,
        verbose=False,
    )

    # creates empty lists to store the loaded images and their labels
    images = []
    labels = []

    # preprocesses each image and stores its matching class label
    for image, label in image_dataset:
        images.append(ka.mobilenet_v2.preprocess_input(image.numpy()))
        labels.append(int(label.numpy()))

    # saves the class names so the label numbers can be checked later
    load_data.class_names = tuple(image_dataset.class_names)

    # returns images and labels as numpy arrays for the next tasks
    return np.asarray(images, dtype=np.float32), np.asarray(labels, dtype=np.int64)

In [5]:
X, Y = load_data('small_flower_dataset')
X.shape, Y.shape, load_data.class_names

((1000, 224, 224, 3),
 (1000,),
 ('daisy', 'dandelion', 'roses', 'sunflowers', 'tulips'))

## Task 3
Prepare your training, validation and test sets for the non-accelerated version of transfer learning.

In [6]:
def split_data(X, Y, train_fraction, randomize=False, eval_set=True):
    """
    Split the data into training and testing sets. If eval_set is True, also create
    an evaluation dataset. There should be two outputs if eval_set there should
    be three outputs (train, test, eval), otherwise two outputs (train, test).
    
    To see what type train, test, and eval should be, refer to the inputs of 
    transfer_learning().
    
    Insert a more detailed description here.

    Returns:
        train_set: tuple of training images and labels in the form (X_train, Y_train)
        eval_set: tuple of validation images and labels in the form (X_eval, Y_eval), if eval_set is True
        test_set: tuple of testing images and labels in the form (X_test, Y_test)
    """
    # checks that the image data and labels contain the same number of samples
    X = np.asarray(X)
    Y = np.asarray(Y)

    if len(X) != len(Y):
        raise ValueError('X and Y must contain the same number of samples')

    if train_fraction <= 0 or train_fraction >= 1:
        raise ValueError('train_fraction must be between 0 and 1')

    # creates empty lists to store the split image and label groups
    train_images = []
    train_labels = []
    eval_images = []
    eval_labels = []
    test_images = []
    test_labels = []

    # splits each class separately so the final sets stay balanced
    for class_label in np.unique(Y):
        # joins the image data and labels together so they stay matched when shuffled
        class_indices = np.where(Y == class_label)[0]

        # shuffles the dataset only when randomize is set to True
        if randomize:
            class_indices = np.random.permutation(class_indices)

        # calculates how many samples should go into the training set
        train_count = int(len(class_indices) * train_fraction)

        if train_count == 0 or train_count == len(class_indices):
            raise ValueError('train_fraction creates an empty train or test split')

        train_indices = class_indices[:train_count]
        remaining_indices = class_indices[train_count:]

        train_images.append(X[train_indices])
        train_labels.append(Y[train_indices])

        # uses the remaining samples for the test set when no evaluation set is needed
        if not eval_set:
            test_indices = remaining_indices

        # splits the remaining samples into validation and test sets when eval_set is True
        else:
            eval_count = len(remaining_indices) // 2
            eval_indices = remaining_indices[:eval_count]
            test_indices = remaining_indices[eval_count:]

            eval_images.append(X[eval_indices])
            eval_labels.append(Y[eval_indices])

        test_images.append(X[test_indices])
        test_labels.append(Y[test_indices])

    # combines each class split back into one training set and one testing set
    X_train = np.concatenate(train_images, axis=0)
    Y_train = np.concatenate(train_labels, axis=0)
    X_test = np.concatenate(test_images, axis=0)
    Y_test = np.concatenate(test_labels, axis=0)

    # combines the validation groups only when an evaluation set is needed
    if eval_set:
        X_eval = np.concatenate(eval_images, axis=0)
        Y_eval = np.concatenate(eval_labels, axis=0)

    # shuffles the final sets so the classes are not grouped together
    if randomize:
        train_order = np.random.permutation(len(Y_train))
        test_order = np.random.permutation(len(Y_test))

        X_train = X_train[train_order]
        Y_train = Y_train[train_order]
        X_test = X_test[test_order]
        Y_test = Y_test[test_order]

        if eval_set:
            eval_order = np.random.permutation(len(Y_eval))
            X_eval = X_eval[eval_order]
            Y_eval = Y_eval[eval_order]

    # returns the split data as tuples in the format required by transfer_learning()
    train_set = (X_train, Y_train)
    test_set = (X_test, Y_test)

    if eval_set:
        validation_set = (X_eval, Y_eval)
        return train_set, validation_set, test_set

    return train_set, test_set

In [7]:
train_set, eval_set, test_set = split_data(X, Y, 0.7, randomize=True, eval_set=True)

Report: Include details of how you have split the data to perform this training. Ensure the split is reasonable and does not introduce class imbalance during training

- The dataset was split after loading the images and labels using:

```python
train_set, eval_set, test_set = split_data(X, Y, 0.7, randomize=True, eval_set=True)
```

- A 70% / 15% / 15% split was used.
- 70% of the data was used for training.
- 15% of the data was used for validation/evaluation.
- 15% of the data was used for testing.
- Since the dataset contains 1000 images across 5 classes, this gives 700 training images, 150 validation images, and 150 testing images.
- Each class has 200 images, so the split keeps the classes balanced.
- Each class has 140 training images, 30 validation images, and 30 testing images.
- The split is done separately for each class before combining the data back together.
- This prevents class imbalance because each flower type contributes the same proportion of images to each set.
- The training set is used to update the model weights.
- The validation/evaluation set is used during training to monitor performance on unseen data.
- The test set is kept separate and should only be used after training to measure final model performance.
- `randomize=True` shuffles the image order so the model does not train on images grouped by class.
- The function returns each split as a tuple in the form `(images, labels)`.
- This format matches the expected input format for the later `transfer_learning()` function.


## Task 4
Using the tf.keras.applications module download a pretrained MobileNetV2 network. 

In [8]:
def load_model():
    '''
    Load a pretrained MobileNetV2 model and return it.

    The model uses ImageNet weights and the same 224x224 RGB input shape
    used when the flower images were loaded in Task 2. The original
    ImageNet classification head is kept here because Task 5 replaces the
    final layer with a Dense layer for the five flower classes.

    Returns:
        model: pretrained MobileNetV2 model loaded with ImageNet weights
    '''
    model = ka.MobileNetV2(
        weights='imagenet',
        include_top=True,
        input_shape=(224, 224, 3),
    )

    # returns the downloaded model so it can be passed into transfer_learning()
    return model

In [9]:
model = load_model()
model.summary()

Model: "mobilenetv2_1.00_224"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 3,538,984 (13.50 MB)

 Trainable params: 3,504,872 (13.37 MB)

 Non-trainable params: 34,112 (133.25 KB)

## Task 5
Replace the last layer of the downloaded neural network with a Dense layer of the appropriate shape for the 5 classes of the small flower dataset {(x1,t1), (x2,t2),..., (xm,tm)}.

## Task 6
Compile and train your model with an SGD optimizer using the following parameters learning_rate=0.01, momentum=0.0, nesterov=False. (NB: The SGD class description can be found at https://keras.io/api/optimizers/sgd/  )

In [10]:
def replace_classification_layer(model, num_classes=5):
    '''
    Replace the pretrained MobileNetV2 ImageNet classifier with a Dense
    layer for the flower dataset classes.

    Inputs:
        - model: pretrained MobileNetV2 model returned by load_model()
        - num_classes: number of flower classes to classify

    Outputs:
        - model: keras.Model with the original MobileNetV2 input and a
            num_classes-unit Dense softmax output layer
    '''
    # keeps the original MobileNetV2 input so Task 2 images still match the model
    model_input = model.input

    # takes the feature vector from the layer before the original ImageNet classifier
    feature_output = model.layers[-2].output

    # creates one softmax probability output for each flower class
    flower_output = keras.layers.Dense(
        num_classes,
        activation='softmax',
        name='flower_predictions',
    )(feature_output)

    # rebuilds the network using the pretrained feature extractor and new classifier
    flower_model = keras.Model(
        inputs=model_input,
        outputs=flower_output,
        name='mobilenetv2_flower_classifier',
    )

    # freezes the pretrained MobileNetV2 layers so only the new Dense head trains later
    for layer in flower_model.layers[:-1]:
        layer.trainable = False

    # keeps the replacement classifier trainable for Task 6
    flower_model.layers[-1].trainable = True

    # Outputs: return the model with a 5-class flower prediction layer
    return flower_model


def transfer_learning(train_set, eval_set, model, parameters):
    '''
    Implement and perform standard transfer learning here.

    Inputs:
        - train_set: list or tuple of the training images and labels in the
            form (images, labels) for training the classifier
        - eval_set: list or tuple of the images and labels used in evaluating
            the model during training, in the form (images, labels)
        - model: an instance of tf.keras.applications.MobileNetV2
        - parameters: list or tuple of parameters to use during training:
            (learning_rate, momentum, nesterov)


    Outputs:
        - model : keras.Model with MobileNetV2 features and a 5-class Dense
            softmax output layer for the flower dataset

    '''
    # Task 5: replace the original 1000-class ImageNet layer with a 5-class flower layer
    model = replace_classification_layer(model, num_classes=5)

    # unpacks the required SGD settings from the parameters tuple
    learning_rate, momentum, nesterov = parameters

    # creates the SGD optimizer required for Task 6
    optimizer = keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=momentum,
        nesterov=nesterov,
    )

    # compiles the model for integer class labels and multi-class prediction
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    # separates the image arrays and class labels for Keras model.fit()
    train_images, train_labels = train_set
    eval_images, eval_labels = eval_set

    # trains only the new flower classifier while monitoring validation performance
    history = model.fit(
        train_images,
        train_labels,
        validation_data=(eval_images, eval_labels),
        epochs=10,
        batch_size=32,
        verbose=1,
    )

    # stores the fit history on the model so Task 7 can plot loss and accuracy curves
    model.training_history = history

    # Outputs: return the compiled and trained 5-class flower classifier
    return model

In [11]:
model = transfer_learning(train_set, eval_set, model, (0.01, 0.0, False))
model.summary()

Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 283ms/step - accuracy: 0.4586 - loss: 1.3467 - val_accuracy: 0.6933 - val_loss: 0.9466
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 268ms/step - accuracy: 0.7414 - loss: 0.7978 - val_accuracy: 0.7933 - val_loss: 0.7112
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 265ms/step - accuracy: 0.7900 - loss: 0.6145 - val_accuracy: 0.8067 - val_loss: 0.6207
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 267ms/step - accuracy: 0.8400 - loss: 0.5220 - val_accuracy: 0.8333 - val_loss: 0.5598
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 265ms/step - accuracy: 0.8571 - loss: 0.4558 - val_accuracy: 0.8333 - val_loss: 0.5206
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 254ms/step - accuracy: 0.8843 - loss: 0.4072 - val_accuracy: 0.8533 - val_loss: 0.4938
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 250ms/step - accuracy: 0.8929 - loss: 0.3725 - val_accuracy: 0.8400 - val_loss: 0.4788
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 250ms/step - accuracy: 0.9057 - loss: 0.3420 - val_accuracy: 0.

Model: "mobilenetv2_flower_classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,264,391 (8.64 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 2 (12.00 B)

## Task 7
Plot the training and validation errors and accuracies of standard transfer 

In [12]:
## Your Code

## Task 8
Experiment with 3 different orders of magnitude for the learning rate. Plot the results and discuss in the below markdown cell

In [13]:
## Your code

### Task 8 Analysis and discussion


## Task 9
Run the resulting classifier on your test dataset using results from the best learning rate you experimented with. Compute and display the confusion matrix. 

In [14]:
## Your code

## Task 10
Compute the precision, recall, and f1 scores of your classifier on the test dataset using the best learning rate. Report on the results and comment. 

In [15]:
## Your code

## Task 11
Perform k-fold validation on the dataset with k = 3. 

In [16]:
def k_fold_validation(features, ground_truth, classifier, k=2):
    '''
    Inputs:
        - features: np.ndarray of features in the dataset
        - ground_truth: np.ndarray of class values associated with the features
        - fit_func: f
        - classifier: class object with both fit() and predict() methods which
        can be applied to subsets of the features and ground_truth inputs.
        - predict_func: function, calling predict_func(features) should return
        a numpy array of class predictions which can in turn be input to the 
        functions in this script to calculate performance metrics.
        - k: int, number of sub-sets to partition the data into. default is k=2
    Outputs:
        - avg_metrics: np.ndarray of shape (3, c) where c is the number of classes.
        The first row is the average precision for each class over the k
        validation steps. Second row is recall and third row is f1 score.
        - sigma_metrics: np.ndarray, each value is the standard deviation of 
        the performance metrics [precision, recall, f1_score]
    '''
    
    #split data
    ### YOUR CODE HERE ###
    
    #go through each partition and use it as a test set.
    for partition_no in range(k):
        #determine test and train sets
        ### YOUR CODE HERE###
        
        #fit model to training data and perform predictions on the test set
        classifier.fit(train_features, train_classes)
        predictions = classifier.predict(test_features)
        
        #calculate performance metrics
        ### YOUR CODE HERE###
    
    #perform statistical analyses on metrics
    ### YOUR CODE HERE###
    
    raise NotImplementedError
    return avg_metrics, sigma_metrics

In [17]:
## Your code
# xx = k_fold_validation(xx, xx, xx, xx)

Comment on the results and any differences with the previous test-train split. 
Repeat with two different values for k and comment on the results. 

### Comments and analysis

## Task 12
With the best learning rate that you found in the previous task, add a non-zero momentum to the training with the SGD optimizer (consider 3 values for the momentum). Report on how your results change.  

In [18]:
## Code

### Report

## Task 13
Now using “accelerated transfer learning”, repeat the training process (k-fold validation is optional this time). You should prepare your training, validation and test sets based on {(F(x1).t1), (F(x2),t2),...,(F(xm),tm)}, and re-do Task 12. 


In [19]:
def accelerated_learning(train_set, eval_set, model, parameters):
    '''
    Implement and perform accelerated transfer learning here.

    Inputs:
        - train_set: list or tuple of the training images and labels in the
            form (images, labels) for training the classifier
        - eval_set: list or tuple of the images and labels used in evaluating
            the model during training, in the form (images, labels)
        - model: an instance of tf.keras.applications.MobileNetV2
        - parameters: list or tuple of parameters to use during training:
            (learning_rate, momentum, nesterov)


    Outputs:
        - model : an instance of tf.keras.applications.MobileNetV2

    '''
    raise NotImplementedError
    return model


Plot and comment on the results and differences against the standard implementation of transfer learning. 

In [ ]:
## Test 


### Your Comments:

## Task 14
Use the results of all experiments to make suggestions for future work and recommendations for parameter values to anyone else who may be interested in a similar implementation of transfer learning. 

### Your answer: